## Imports

In [1]:
import pandas as pd
import os
from pathlib import Path
from src.utils import config
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS



PROJECT_ROOT = Path.cwd()
print('PROJECT_ROOT:', PROJECT_ROOT)

DATA_ROOT = os.path.join(PROJECT_ROOT, "data")
data_resources_dir = os.path.join(DATA_ROOT, "resources")
data_pat_cases_dir = os.path.join(DATA_ROOT, "patients")


PROJECT_ROOT: /Users/if66gb/Dropbox/MyProgramming/PythonProjects/PyCharm/summerschool-ails2026-clinicalNLP


## Read/load data

In [2]:
# Read the CCC taxonomy files, diagnosis and interventions
ccc_diag_df = pd.read_csv(os.path.join(data_resources_dir, "CCC_Diagnosisv25-20120309_sorted.csv"), encoding="utf-8-sig")
ccc_interv_df = pd.read_csv(os.path.join(data_resources_dir, "CCC_Interventionsv25-20109_sorted.csv"), encoding="utf-8-sig")


print('ccc_diag_df.shape:', ccc_diag_df.shape)
print(ccc_diag_df.head())

print('ccc_interv_df.shape:', ccc_interv_df.shape)
print(ccc_interv_df.head())

ccc_diag_df.shape: (176, 4)
  CompLetter  Code                          Name  \
0          A   1.0           Activity Alteration   
1          A   1.1          Activity Intolerance   
2          A   1.2     Activity Intolerance Risk   
3          A   1.3  Diversional Activity Deficit   
4          A   1.4                       Fatigue   

                                         Description  
0  Change in or modification of energy used by th...  
1  Incapacity to carry out physiological or psych...  
2  Increased chance of an incapacity to carry out...  
3  Lack of interest or engagement in leisure acti...  
4  Exhaustion that interferes with physical and m...  
ccc_interv_df.shape: (201, 4)
  CompLetter  Code                 Name  \
0          A   1.0        Activity Care   
1          A   1.2  Energy Conservation   
2          A   2.0        Fracture Care   
3          A   2.1            Cast Care   
4          A   2.2     Immobilizer Care   

                                        

In [3]:
# Read ICD-10 file
icd10_df = pd.read_csv(os.path.join(data_resources_dir, "ICD10_clean.csv"), encoding="utf-8-sig")

print('icd10_df.shape:', icd10_df.shape)
print(icd10_df.head())

icd10_df.shape: (11539, 4)
      code      kind parent_code  \
0        I   chapter         NaN   
1  A00-A09     block           I   
2      A00  category     A00-A09   
3    A00.0  category         A00   
4    A00.1  category         A00   

                                               title  
0          Certain infectious and parasitic diseases  
1                     Intestinal infectious diseases  
2                                            Cholera  
3  Cholera due to Vibrio cholerae 01, biovar chol...  
4    Cholera due to Vibrio cholerae 01, biovar eltor  


In [4]:
# Read Care guidelines
ccg_df = pd.read_csv(os.path.join(data_resources_dir, "Care_guidelines.csv"), encoding="utf-8-sig")

print('ccg_df:', ccg_df.shape)
print(ccg_df.head())

ccg_df: (96, 5)
  identifier  sortkey language  \
0   ccs00013   50.041       en   
1   ccs00012   50.042       en   
2   ccs00038   35.050       en   
3   ccs00004   50.022       en   
4   ccs00010   50.036       en   

                                               title  \
0                                         Drug abuse   
1                                Parkinson's disease   
2                                      Schizophrenia   
3  Venous thromboembolism (VTE):  deep venous thr...   
4                                Atrial fibrillation   

                                             content  
0  The Finnish treatment guarantee includes peopl...  
1  The treatment of Parkinson’s disease may be in...  
2  Schizophrenia is a severe psychiatric disorder...  
3  Key recommendations in management of VTE inclu...  
4  The prevalence and incidence of atrial fibrill...  


In [5]:
# Read patient cases with clinical notes
pat1_df = pd.read_csv(os.path.join(data_pat_cases_dir, "Case1_Left_Ventricular_Fibroma.csv"), encoding="utf-8-sig")
pat2_df = pd.read_csv(os.path.join(data_pat_cases_dir, "Case2_Postoperative_Chylous_Ascites.csv"), encoding="utf-8-sig")
pat3_df = pd.read_csv(os.path.join(data_pat_cases_dir, "Case3_Multifocal_Schwannoma.csv"), encoding="utf-8-sig")
pat4_df = pd.read_csv(os.path.join(data_pat_cases_dir, "Case4_Acute_Limb_Ischemia.csv"), encoding="utf-8-sig")
pat5_df = pd.read_csv(os.path.join(data_pat_cases_dir, "Case5_Bronchopleural_Fistula.csv"), encoding="utf-8-sig")


pat1_df = pat1_df.sort_values(["day", "note_order"])
pat2_df = pat2_df.sort_values(["day", "note_order"])
pat3_df = pat3_df.sort_values(["day", "note_order"])
pat4_df = pat4_df.sort_values(["day", "note_order"])
pat5_df = pat5_df.sort_values(["day", "note_order"])

pat_all_df = pd.concat([pat1_df, pat2_df, pat3_df, pat4_df, pat5_df], ignore_index=True)
pat_all_df = pat_all_df.sort_values(["case_id", "day", "note_order"]).reset_index(drop=True)

print('pat1_df:', pat1_df.shape)
print(pat1_df.head())

pat1_df: (25, 7)
   case_id                 case_name  day  note_order  \
0        1  Left Ventricular Fibroma   -7           1   
1        1  Left Ventricular Fibroma   -5           2   
2        1  Left Ventricular Fibroma   -4           3   
3        1  Left Ventricular Fibroma    0           4   
4        1  Left Ventricular Fibroma    0           5   

                         note_title                                    author  \
0      Outpatient Clinic Evaluation                    Pediatric Cardiologist   
1           Echocardiography Report  Cardiac Imaging Specialist / Radiologist   
2             Surgical Consultation                    Cardiothoracic Surgeon   
3            Admission (Cardiology)                    Pediatric Cardiologist   
4  Admission Nursing Note (Evening)                   Pediatric Cardiac Nurse   

                                           note_text  
0  Previously healthy child referred after abnorm...  
1  TTE shows well-defined echogenic mass in

## Create vector indices

In [6]:
def init_embedding_model(embedding_model_name):
    emb_model = OpenAIEmbeddings(model=embedding_model_name, api_key=config.get_api_key())
    return emb_model


def init_llm(llm_name):
    llm = ChatOpenAI(model=llm_name, api_key=config.get_api_key())
    return llm


def load_index(filepath, emb_model):
    vector_store = FAISS.load_local(
        filepath,
        emb_model,
        allow_dangerous_deserialization=True,
    )
    return vector_store


def query_index(vector_store, query_str, top_k=5):
    res = vector_store.similarity_search_with_score(query_str, k=top_k)
    for doc, score in res:
        print(f'Sim: {score:.2f}. Content: "{' '.join(doc.page_content.split())}". Metadata: {doc.metadata}')



In [7]:
# LLM API services
embedding_model_name = "text-embedding-3-small"
llm_name = "gpt-5.6-luna"

VEC_STORE_DIR = os.path.join(PROJECT_ROOT, "vector_stores")

emb = init_embedding_model(embedding_model_name)
llm = init_llm(llm_name)


### Load and query the patient index

In [8]:
pat1_faiss_index_path = os.path.join(VEC_STORE_DIR, "pat1_faiss_index")
pat2_faiss_index_path = os.path.join(VEC_STORE_DIR, "pat2_faiss_index")
pat3_faiss_index_path = os.path.join(VEC_STORE_DIR, "pat3_faiss_index")
pat4_faiss_index_path = os.path.join(VEC_STORE_DIR, "pat4_faiss_index")
pat5_faiss_index_path = os.path.join(VEC_STORE_DIR, "pat5_faiss_index")
pat6_faiss_index_path = os.path.join(VEC_STORE_DIR, "pat6_faiss_index")

# Load existing vector store
pat_all_vector_store = load_index(filepath=pat1_faiss_index_path, emb_model=emb)


query = "The patient slipped and needed some help due to fractured toe"
tok_k = 5
print(f'Query: "{query}"')
print(f'\nRetrieved results (top {tok_k}):')
query_index(pat_all_vector_store, query_str=query, top_k=tok_k)

Query: "The patient slipped and needed some help due to fractured toe"

Retrieved results (top 5):
Sim: 0.34. Content: "Ward Nursing Note. Author: Ward Nurse. Continues to improve. Ambulating independently. Pain minimal. Engaging in normal activities. Parents satisfied w/ progress.". Metadata: {'source': 'Patient_Notes', 'case_id': 1, 'case_name': 'Left Ventricular Fibroma', 'day': 5, 'note_order': 19, 'note_title': 'Ward Nursing Note', 'author': 'Ward Nurse'}
Sim: 0.33. Content: "Ward Nursing Note (Morning). Author: Ward Nurse. Awake, interactive, good mood. Vitals stable. Pain 3/10 → oral analgesia given. Assisted w/ hygiene + mobilization. Ambulated short distance. Encouraged deep breathing.". Metadata: {'source': 'Patient_Notes', 'case_id': 1, 'case_name': 'Left Ventricular Fibroma', 'day': 4, 'note_order': 16, 'note_title': 'Ward Nursing Note (Morning)', 'author': 'Ward Nurse'}
Sim: 0.30. Content: "ICU Nursing Note (Post-op Evening). Author: Pediatric ICU Nurse. Received from OR i

### Load and query the care guideline index

In [9]:
ccg_faiss_index_path = os.path.join(VEC_STORE_DIR, "ccg_faiss_index")

# Load existing vector stores
ccg_vector_store = load_index(filepath=ccg_faiss_index_path, emb_model=emb)


query = "The patient slipped and needed some help due to fractured toe"
tok_k = 5
print(f'Query: "{query}"')
print(f'\nRetrieved results (top {tok_k}):')
results_icd10 = query_index(ccg_vector_store, query_str=query, top_k=tok_k)

Query: "The patient slipped and needed some help due to fractured toe"

Retrieved results (top 5):
Sim: 0.39. Content: "Hip fracture: Hip fractures often lead to disability and need for long-term care. To prevent fractures, the risk of falls should be assessed regularly and modifiable risk factors should be eliminated. Hip fracture patients’ care should be seamless and based on comprehensive geriatric assessment. Surgical treatment should enable early mobilization. Cemented fixation is preferred for hip replacements. Multidisciplinary geriatric care and rehabilitation improve outcomes and help to prevent complications, including delirium. Rehabilitation is tailored individually, should include strength training and should continue after discharge. Identifying reasons for fracture is essential for preventing future fractures.". Metadata: {'source': 'Care_Guidelines', 'identifier': 'ccs00092', 'sortkey': '50.04', 'language': 'en', 'title': 'Hip fracture', 'content_type': 'title--content'

### Load and query the CCC indices

In [10]:
ccc_diag_faiss_index_path = os.path.join(VEC_STORE_DIR, "ccc_diag_faiss_index"); ccc_diag_source = "CCC_Diagnosis"
ccc_interv_faiss_index_path = os.path.join(VEC_STORE_DIR, "ccc_interv_faiss_index"); ccc_interv_source = "CCC_Interventions"

# Load existing vector stores
ccc_diag_vector_store = load_index(filepath=ccc_diag_faiss_index_path, emb_model=emb)
ccc_interv_vector_store = load_index(filepath=ccc_interv_faiss_index_path, emb_model=emb)


query = "Patient is simply lazy"
tok_k = 5
print(f'Query: "{query}"')
print(f'\nRetrieved results CCC Diagnoses (top {tok_k}):')
results_ccc_diag = query_index(ccc_diag_vector_store, query_str=query, top_k=tok_k)
print(f'\nRetrieved results CCC Interventions (top {tok_k}):')
results_ccc_interv = query_index(ccc_interv_vector_store, query_str=query, top_k=tok_k)

Query: "Patient is simply lazy"

Retrieved results CCC Diagnoses (top 5):
Sim: 0.43. Content: "Self Care Deficit". Metadata: {'source': 'CCC_Diagnosis', 'comp_letter': 'O', 'code': 38.0, 'content_type': 'name'}
Sim: 0.39. Content: "Disuse Syndrome". Metadata: {'source': 'CCC_Diagnosis', 'comp_letter': 'N', 'code': 33.2, 'content_type': 'name'}
Sim: 0.38. Content: "Lack of interest or engagement in leisure activities". Metadata: {'source': 'CCC_Diagnosis', 'comp_letter': 'A', 'code': 1.3, 'content_type': 'description'}
Sim: 0.38. Content: "Failure to follow regulated course of treating disease". Metadata: {'source': 'CCC_Diagnosis', 'comp_letter': 'G', 'code': 20.6, 'content_type': 'description'}
Sim: 0.38. Content: "Knowledge Deficit of Medication Regimen". Metadata: {'source': 'CCC_Diagnosis', 'comp_letter': 'D', 'code': 8.5, 'content_type': 'name'}

Retrieved results CCC Interventions (top 5):
Sim: 0.40. Content: "Bedbound Care". Metadata: {'source': 'CCC_Interventions', 'comp_letter

### Load and query the ICD-10 code index

In [11]:
icd10_faiss_index_path = os.path.join(VEC_STORE_DIR, "icd10_faiss_index")

# Load existing vector store
icd10_vector_store = load_index(filepath=icd10_faiss_index_path, emb_model=emb)


query = "SSRI" #"The patient slipped"
tok_k = 5
print(f'\nTesting semantic search. Query: "{query}"')
print(f'Search in ICD-10, top {tok_k}:')
results_icd10 = query_index(icd10_vector_store, query_str=query, top_k=tok_k)


Testing semantic search. Query: "SSRI"
Search in ICD-10, top 5:
Sim: 0.55. Content: "Y49.1: Monoamine-oxidase-inhibitor antidepressants". Metadata: {'source': 'ICD10', 'code': 'Y49.1', 'kind': 'category', 'parent_code': 'Y49', 'content_type': 'code--title'}
Sim: 0.53. Content: "Monoamine-oxidase-inhibitor antidepressants". Metadata: {'source': 'ICD10', 'code': 'T43.1', 'kind': 'category', 'parent_code': 'T43', 'content_type': 'title'}
Sim: 0.53. Content: "Monoamine-oxidase-inhibitor antidepressants". Metadata: {'source': 'ICD10', 'code': 'Y49.1', 'kind': 'category', 'parent_code': 'Y49', 'content_type': 'title'}
Sim: 0.52. Content: "Other and unspecified antidepressants". Metadata: {'source': 'ICD10', 'code': 'T43.2', 'kind': 'category', 'parent_code': 'T43', 'content_type': 'title'}
Sim: 0.52. Content: "Other and unspecified antidepressants". Metadata: {'source': 'ICD10', 'code': 'Y49.2', 'kind': 'category', 'parent_code': 'Y49', 'content_type': 'title'}


## Simple RAG-based Q&A system

In [14]:
from src.rag.rag import load_vector_stores, rag_q_and_a

# --------------------------------------------------------
# Load vector store dict
# --------------------------------------------------------
vector_store_dict = load_vector_stores(
    vec_store_dir=VEC_STORE_DIR,
    emb_model=emb,
)

print(
    "Loaded vector stores:",
    list(vector_store_dict.keys()),
)


##########################################################
# --------------------------------------------------------
# Settings for the query loop
# --------------------------------------------------------
search_vector_store_names = [
    #"pat1",
    #"pat2",
    #"pat3",
    #"pat4",
    "pat5",
    #"pat6",
    "icd10",
    "ccc_diag",
    "ccc_interv",
    "ccg",
]
top_k = 5
include_history = True
##########################################################

history = []

#query = "The patient is having fever, coughing, and yellow mucus. What diagnosis could explain the patient's symptoms, and which ICD-10 code is most relevant?"


# --------------------------------------------------------
# Query loop
# --------------------------------------------------------
print("\nClinical RAG system")
print("Vector stores that we search in:", search_vector_store_names)
print("Type 'exit' to quit.\n")

while True:
    query = input(">>> QUESTION: ").strip()
    print("\n>>> QUESTION: ", query)

    if query.lower() in ["exit", "quit", "q"]:
        print("Stopping.")
        break

    if query.lower() in ["/h", "/hist", "/history"]:
        print("\nChat history:")
        print(history)
        print()
        continue

    if not query:
        continue

    answer, results, rewritten_query = rag_q_and_a(
        query=query,
        vector_store_names=search_vector_store_names,
        vector_stores=vector_store_dict,
        llm=llm,
        top_k=top_k,
        history=history,
    )

    print("\nREWRITTEN INDEX SEARCH QUERY:")
    print(rewritten_query)

    print("\nANSWER:")
    print(answer)
    print()

    if include_history:
        # ----------------------------------------------------
        # Update conversation history
        # ----------------------------------------------------
        history.append({
            "role": "user",
            "content": query,
        })

        history.append({
            "role": "assistant",
            "content": answer,
        })

Loaded vector stores: ['ccc_diag', 'ccc_interv', 'ccg', 'icd10', 'pat1', 'pat2', 'pat3', 'pat4', 'pat5', 'pat6']

Clinical RAG system
Vector stores that we search in: ['pat5', 'icd10', 'ccc_diag', 'ccc_interv', 'ccg']
Type 'exit' to quit.



KeyboardInterrupt: Interrupted by user